# كشكول التجارب: دليل اتخاذ القرار - متى نستخدم ومتى نتجنب MMR؟ (المحاضرة 48)

أهلاً بيك في معمل المحاضرة 48!
مش دايماً MMR هو الحل السحري لكل حاجة.
في المعمل ده هنتعلم عملياً:
- إمتى الـ MMR يكون عبقري ومفيش بديل عنه؟ (زي استكشاف وتلخيص موضوع كبير فيه آراء أو أبعاد متعددة).
- وإمتى الـ MMR يكون فكرة سيئة جداً ويقلل دقة الإجابة؟ (زي لما تسأل عن رقم فاتورة محدد أو منفذ بورت أو كود خطأ).

### 1. تجهيز الفهرس والمستندات
يلا نجهز وثائق فيها موضوعات عامة وفيها كمان حقائق دقيقة محددة:

In [1]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

data = [
    Document(page_content="من أهم مميزات المعمارية المصغرة Microservices إمكانية نشر كل خدمة وتوسيعها بشكل مستقل."),
    Document(page_content="من أكبر عيوب المعمارية المصغرة Microservices تعقيد الشبكات وصعوبة تتبع الأخطاء بين الخدمات الموزعة."),
    Document(page_content="منفذ الاتصال الافتراضي لقاعدة بيانات PostgreSQL هو المنفذ 5432."),
    Document(page_content="منفذ الاتصال الافتراضي لقاعدة بيانات MySQL هو المنفذ 3306."),
]

vector_store = FAISS.from_documents(data, embeddings)
print("الفهرس جاهز للتجربة!")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_20404\4136968265.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

الفهرس جاهز للتجربة!


### 2. سيناريو (أ): السؤال الاستكشافي العام - تفوق MMR الكاسح
تعال نسأل سؤال عن مميزات وعيوب المعمارية:

In [2]:
query_broad = "ما هي مميزات وتحديات المايكروسيرفس؟"

# البحث العادي
sim_res = vector_store.similarity_search(query_broad, k=2)
print("نتائج البحث العادي:")
for d in sim_res:
    print(f"- {d.page_content}")

# بحث MMR
mmr_res = vector_store.max_marginal_relevance_search(query_broad, k=2, lambda_mult=0.5)
print("\nنتائج بحث MMR المتنوعة:")
for d in mmr_res:
    print(f"- {d.page_content}")

نتائج البحث العادي:
- من أهم مميزات المعمارية المصغرة Microservices إمكانية نشر كل خدمة وتوسيعها بشكل مستقل.
- من أكبر عيوب المعمارية المصغرة Microservices تعقيد الشبكات وصعوبة تتبع الأخطاء بين الخدمات الموزعة.

نتائج بحث MMR المتنوعة:
- من أهم مميزات المعمارية المصغرة Microservices إمكانية نشر كل خدمة وتوسيعها بشكل مستقل.
- منفذ الاتصال الافتراضي لقاعدة بيانات PostgreSQL هو المنفذ 5432.


### 3. سيناريو (ب): السؤال عن حقيقة محددة - تجنب المبالغة في التنوع
لما نسأل عن رقم المنفذ لقاعدة بوستجريس:

In [3]:
query_fact = "ما هو رقم البورت الافتراضي لبوستجريس؟"

# البحث العادي المباشر
direct_res = vector_store.similarity_search(query_fact, k=1)
print(f"النتيجة الدقيقة المباشرة: {direct_res[0].page_content}")

النتيجة الدقيقة المباشرة: منفذ الاتصال الافتراضي لقاعدة بيانات MySQL هو المنفذ 3306.


### 4. جرب بنفسك!
جرب تسأل عن المنفذ بتاع MySQL واختبر هل إجبار النظام على تنوع مفرط (lambda=0.0) مفيد ولا مشتت في حالة الأرقام المحددة.